In [0]:
from pyspark.sql.functions import *

### creating the silver schema

In [0]:
%sql
create schema if not exists de_project.silver;

### reading the raw data in silver from bronze schema

In [0]:
posts_df = spark.table("de_project.bronze.raw_posts")
comments_df = spark.table("de_project.bronze.raw_comments")

### removing the dublicates

In [0]:
posts_df.dropDuplicates()

DataFrame[userId: bigint, id: bigint, title: string, body: string, event_time: timestamp, Current_timestamp: timestamp, ingest_time: timestamp]

In [0]:
comments_df.dropDuplicates()

DataFrame[postId: bigint, id: bigint, name: string, email: string, body: string, ingest_time: timestamp]

### cleans body text to lowercase trimmed

In [0]:
posts_df = posts_df.withColumn("body_clean", lower(trim(col("body"))))

### generating random past timestamp column

In [0]:
from pyspark.sql.functions import expr

posts_df = posts_df.withColumn(
    "log_timestamp",
    expr("from_utc_timestamp(current_timestamp(), 'Asia/Kolkata') - (rand() * 86400) * INTERVAL 1 SECOND")
)

### assign the log level based on eror words

In [0]:

posts_df = posts_df.withColumn(
    "log_level",
    when(col("body_clean").rlike("error"), "ERROR")
    .when(col("body_clean").rlike("fail"), "ERROR")
    .when(col("body_clean").rlike("warn"), "WARN")
    .otherwise("INFO")
)

### genrating the flag for error

In [0]:
posts_df = posts_df.withColumn(
    "is_error",
    when(col("log_level") == "ERROR", 1).otherwise(0)
)

In [0]:
posts_df = posts_df.withColumn(
    "user_id",
    col("userId")
)

### extracting date, hour and adding current pipling timestamp

In [0]:
posts_df = posts_df.withColumn("log_date", to_date("log_timestamp")) \
                   .withColumn("log_hour", hour("log_timestamp"))\
                       .withColumn("Current_timestamp", from_utc_timestamp(current_timestamp(), 'Asia/Kolkata'))

In [0]:
posts_silver = posts_df.select(
    "id",
    "user_id",
    "title",
    "body_clean",
    "log_timestamp",
    "log_date",
    "log_hour",
    "log_level",
    "is_error",
    "Current_timestamp"
)

In [0]:
display(posts_silver)

id,user_id,title,body_clean,log_timestamp,log_date,log_hour,log_level,is_error,Current_timestamp
1,1,sunt aut facere repellat provident occaecati excepturi optio reprehenderit,quia et suscipit suscipit recusandae consequuntur expedita et cum reprehenderit molestiae ut ut quas totam nostrum rerum est autem sunt rem eveniet architecto,2026-04-06T01:49:24.826713Z,2026-04-06,1,INFO,0,2026-04-06T14:54:29.006652Z
2,1,qui est esse,est rerum tempore vitae sequi sint nihil reprehenderit dolor beatae ea dolores neque fugiat blanditiis voluptate porro vel nihil molestiae ut reiciendis qui aperiam non debitis possimus qui neque nisi nulla,2026-04-05T23:00:54.032938Z,2026-04-05,23,INFO,0,2026-04-06T14:54:29.006652Z
3,1,ea molestias quasi exercitationem repellat qui ipsa sit aut,et iusto sed quo iure voluptatem occaecati omnis eligendi aut ad voluptatem doloribus vel accusantium quis pariatur molestiae porro eius odio et labore et velit aut,2026-04-06T06:16:05.878226Z,2026-04-06,6,INFO,0,2026-04-06T14:54:29.006652Z
4,1,eum et est occaecati,ullam et saepe reiciendis voluptatem adipisci sit amet autem assumenda provident rerum culpa quis hic commodi nesciunt rem tenetur doloremque ipsam iure quis sunt voluptatem rerum illo velit,2026-04-06T11:48:34.654996Z,2026-04-06,11,INFO,0,2026-04-06T14:54:29.006652Z
5,1,nesciunt quas odio,repudiandae veniam quaerat sunt sed alias aut fugiat sit autem sed est voluptatem omnis possimus esse voluptatibus quis est aut tenetur dolor neque,2026-04-05T20:31:10.577981Z,2026-04-05,20,INFO,0,2026-04-06T14:54:29.006652Z
6,1,dolorem eum magni eos aperiam quia,ut aspernatur corporis harum nihil quis provident sequi mollitia nobis aliquid molestiae perspiciatis et ea nemo ab reprehenderit accusantium quas voluptate dolores velit et doloremque molestiae,2026-04-06T14:30:36.586012Z,2026-04-06,14,INFO,0,2026-04-06T14:54:29.006652Z
7,1,magnam facilis autem,dolore placeat quibusdam ea quo vitae magni quis enim qui quis quo nemo aut saepe quidem repellat excepturi ut quia sunt ut sequi eos ea sed quas,2026-04-05T15:17:16.956344Z,2026-04-05,15,INFO,0,2026-04-06T14:54:29.006652Z
8,1,dolorem dolore est ipsam,dignissimos aperiam dolorem qui eum facilis quibusdam animi sint suscipit qui sint possimus cum quaerat magni maiores excepturi ipsam ut commodi dolor voluptatum modi aut vitae,2026-04-06T08:30:24.935362Z,2026-04-06,8,INFO,0,2026-04-06T14:54:29.006652Z
9,1,nesciunt iure omnis dolorem tempora et accusantium,consectetur animi nesciunt iure dolore enim quia ad veniam autem ut quam aut nobis et est aut quod aut provident voluptas autem voluptas,2026-04-05T19:01:20.264019Z,2026-04-05,19,INFO,0,2026-04-06T14:54:29.006652Z
10,1,optio molestias id quia eum,quo et expedita modi cum officia vel magni doloribus qui repudiandae vero nisi sit quos veniam quod sed accusamus veritatis error,2026-04-06T04:50:07.112337Z,2026-04-06,4,ERROR,1,2026-04-06T14:54:29.006652Z


### storing the clene data in silver schema

In [0]:
posts_silver.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable("de_project.silver.posts_clean")

### Starting for comment data

In [0]:
comments_df = comments_df.withColumn("body_clean", lower(trim(col("body"))))

### cleans email to lowercase trimmed

In [0]:
comments_df = comments_df.withColumn("email_clean", lower(trim(col("email"))))

### extracting domain 

In [0]:
comments_df = comments_df.withColumn(
    "email_domain",
    regexp_extract(col("email_clean"), r"@(.+)", 1)
)

### extracting the username

In [0]:
comments_df = comments_df.withColumn(
    "email_user",
    regexp_extract(col("email_clean"), r"(^[^@]+)", 1)
)

### generating random past timestamp column

In [0]:
from pyspark.sql.functions import expr

comments_df = comments_df.withColumn(
    "log_timestamp",
    expr("current_timestamp() - (rand() * 864000) * INTERVAL 1 SECOND")
)

### assign the log level based on eror words

In [0]:
comments_df = comments_df.withColumn(
    "log_level",
    when(col("body_clean").rlike("error"), "ERROR")
    .when(col("body_clean").rlike("fail"), "ERROR")
    .when(col("body_clean").rlike("warn"), "WARN")
    .otherwise("INFO")
)

In [0]:
#flag genarting for error

comments_df = comments_df.withColumn(
    "is_error",
    when(col("log_level") == "ERROR", 1).otherwise(0)
)

In [0]:
#extracting date, hour and adding current pipling timestamp

comments_df = comments_df.withColumn("log_date", to_date("log_timestamp")) \
                         .withColumn("log_hour", hour("log_timestamp"))\
                             .withColumn("Current_timestamp", from_utc_timestamp(current_timestamp(), 'Asia/Kolkata'))

In [0]:
comments_silver = comments_df.select(
    "id",
    "postId",
    "email_clean",
    "email_user",
    "email_domain",
    "body_clean",
    "log_timestamp",
    "log_date",
    "log_hour",
    "log_level",
    "is_error",
    "Current_timestamp"
)

### storing clean comment data in silver schema

In [0]:
comments_silver.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable("de_project.silver.comments_clean")

In [0]:
comments_silver.printSchema()

root
 |-- id: long (nullable = true)
 |-- postId: long (nullable = true)
 |-- email_clean: string (nullable = true)
 |-- email_user: string (nullable = true)
 |-- email_domain: string (nullable = true)
 |-- body_clean: string (nullable = true)
 |-- log_timestamp: timestamp (nullable = false)
 |-- log_date: date (nullable = false)
 |-- log_hour: integer (nullable = false)
 |-- log_level: string (nullable = false)
 |-- is_error: integer (nullable = false)
 |-- Current_timestamp: timestamp (nullable = false)



In [0]:
display(comments_silver)

id,postId,email_clean,email_user,email_domain,body_clean,log_timestamp,log_date,log_hour,log_level,is_error,Current_timestamp
1,1,eliseo@gardner.biz,eliseo,gardner.biz,laudantium enim quasi est quidem magnam voluptate ipsam eos tempora quo necessitatibus dolor quam autem quasi reiciendis et nam sapiente accusantium,2026-03-30T15:59:34.113801Z,2026-03-30,15,INFO,0,2026-04-06T14:54:38.690071Z
2,1,jayne_kuhic@sydney.com,jayne_kuhic,sydney.com,est natus enim nihil est dolore omnis voluptatem numquam et omnis occaecati quod ullam at voluptatem error expedita pariatur nihil sint nostrum voluptatem reiciendis et,2026-04-04T09:12:47.091621Z,2026-04-04,9,ERROR,1,2026-04-06T14:54:38.690071Z
3,1,nikita@garfield.biz,nikita,garfield.biz,quia molestiae reprehenderit quasi aspernatur aut expedita occaecati aliquam eveniet laudantium omnis quibusdam delectus saepe quia accusamus maiores nam est cum et ducimus et vero voluptates excepturi deleniti ratione,2026-04-06T02:08:04.268731Z,2026-04-06,2,INFO,0,2026-04-06T14:54:38.690071Z
4,1,lew@alysha.tv,lew,alysha.tv,non et atque occaecati deserunt quas accusantium unde odit nobis qui voluptatem quia voluptas consequuntur itaque dolor et qui rerum deleniti ut occaecati,2026-04-01T08:30:53.225006Z,2026-04-01,8,INFO,0,2026-04-06T14:54:38.690071Z
5,1,hayden@althea.biz,hayden,althea.biz,harum non quasi et ratione tempore iure ex voluptates in ratione harum architecto fugit inventore cupiditate voluptates magni quo et,2026-03-29T20:06:06.253621Z,2026-03-29,20,INFO,0,2026-04-06T14:54:38.690071Z
6,2,presley.mueller@myrl.com,presley.mueller,myrl.com,doloribus at sed quis culpa deserunt consectetur qui praesentium accusamus fugiat dicta voluptatem rerum ut voluptate autem voluptatem repellendus aspernatur dolorem in,2026-03-28T17:15:16.856198Z,2026-03-28,17,INFO,0,2026-04-06T14:54:38.690071Z
7,2,dallas@ole.me,dallas,ole.me,maiores sed dolores similique labore et inventore et quasi temporibus esse sunt id et eos voluptatem aliquam aliquid ratione corporis molestiae mollitia quia et magnam dolor,2026-03-28T05:59:48.597232Z,2026-03-28,5,INFO,0,2026-04-06T14:54:38.690071Z
8,2,mallory_kunze@marie.org,mallory_kunze,marie.org,ut voluptatem corrupti velit ad voluptatem maiores et nisi velit vero accusamus maiores voluptates quia aliquid ullam eaque,2026-03-31T03:25:44.652656Z,2026-03-31,3,INFO,0,2026-04-06T14:54:38.690071Z
9,2,meghan_littel@rene.us,meghan_littel,rene.us,sapiente assumenda molestiae atque adipisci laborum distinctio aperiam et ab ut omnis et occaecati aspernatur odit sit rem expedita quas enim ipsam minus,2026-03-29T01:49:08.851503Z,2026-03-29,1,INFO,0,2026-04-06T14:54:38.690071Z
10,2,carmen_keeling@caroline.name,carmen_keeling,caroline.name,voluptate iusto quis nobis reprehenderit ipsum amet nulla quia quas dolores velit et non aut quia necessitatibus nostrum quaerat nulla et accusamus nisi facilis,2026-03-30T11:31:27.042372Z,2026-03-30,11,INFO,0,2026-04-06T14:54:38.690071Z
